In [1]:
import mlflow

import pandas as pd
import mlflow.sklearn

from sklearn.feature_extraction.text import CountVectorizer

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

import re
import string

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

import numpy as np

C:\Users\aryan\AppData\Roaming\Python\Python312\site-packages\pydantic\_internal\_fields.py:132: UserWarning: Field "model_name" in PromptModelConfig has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


In [2]:
df = pd.read_csv('IMDB.csv')
df = df.sample(500)
df.to_csv('data.csv', index=False)
df.head()

,review,sentiment
377,Jon Cryer reprises his role as a neurotic guy ...,positive
436,"Not since Bette Davis's 1933 vehicle ""Ex-Lady""...",positive
87,"""Don't be greedy"" sums up the depth of this mo...",negative
563,I really enjoyed this movie and it was a littl...,positive
432,"My God, what an incredible movie it is! Remind...",negative


# data preprocessing

In [3]:
# data preprocessing

# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

<>:32: SyntaxWarning: invalid escape sequence '\s'
<>:32: SyntaxWarning: invalid escape sequence '\s'
C:\Users\aryan\AppData\Local\Temp\ipykernel_25864\3798096103.py:32: SyntaxWarning: invalid escape sequence '\s'
  text = re.sub('\s+', ' ', text).strip()


In [4]:
df = normalize_text(df)
df.head()

,review,sentiment
377,jon cryer reprises role neurotic guy two half ...,positive
436,since bette davis s vehicle ex lady seen film ...,positive
87,don t greedy sum depth movie rest baloney big ...,negative
563,really enjoyed movie little difficult brother ...,positive
432,god incredible movie is reminded much similar ...,negative


In [5]:
df['sentiment'].value_counts()

sentiment
positive    250
negative    250
Name: count, dtype: int64

In [6]:
x = df['sentiment'].isin(['positive','negative'])
df = df[x]

In [7]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})
df.head()

,review,sentiment
377,jon cryer reprises role neurotic guy two half ...,1
436,since bette davis s vehicle ex lady seen film ...,1
87,don t greedy sum depth movie rest baloney big ...,0
563,really enjoyed movie little difficult brother ...,1
432,god incredible movie is reminded much similar ...,0


In [13]:
df.isnull()

,review,sentiment
377,False,False
436,False,False
87,False,False
563,False,False
432,False,False
...,...,...
55,False,False
626,False,False
945,False,False
143,False,False


In [8]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [9]:
vectorizer = CountVectorizer(max_features=100)
X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [ ]:
import dagshub
import mlflow

mlflow.set_tracking_uri('https://dagshub.com/aryan-Patel-web/production-sentiment-analysis-end-to-end-mlops.mlflow')

dagshub.init(repo_owner='aryan-Patel-web', repo_name='production-sentiment-analysis-end-to-end-mlops', mlflow=True)


# mlflow.set_experiment("Logistic Regression Baseline")
mlflow.set_experiment("Logistic Regression Baseline")


Accessing as aryan-Patel-web

Initialized MLflow to track repo "aryan-Patel-web/production-sentiment-analysis-end-to-end-mlops"

Repository aryan-Patel-web/production-sentiment-analysis-end-to-end-mlops initialized!

<Experiment: artifact_location='mlflow-artifacts:/5ccf8f18a02d4d51b2fcd904a29999ac', creation_time=1774520206688, experiment_id='0', last_update_time=1774520206688, lifecycle_stage='active', name='Logistic Regression Baseline', tags={}, workspace='default'>

In [12]:
import mlflow
import logging
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLflow run...")

with mlflow.start_run():
    start_time = time.time()
    
    try:
        logging.info("Logging preprocessing parameters...")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param("num_features", 100)
        mlflow.log_param("test_size", 0.25)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter=1000)  # Increase max_iter to prevent non-convergence issues

        logging.info("Fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model training complete.")

        logging.info("Logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model, "model")

        # Log execution time
        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")

        # Save and log the notebook
        # notebook_path = "exp1_baseline_model.ipynb"
        # logging.info("Executing Jupyter Notebook. This may take a while...")
        # os.system(f"jupyter nbconvert --to notebook --execute --inplace {notebook_path}")
        # mlflow.log_artifact(notebook_path)

        # logging.info("Notebook execution and logging complete.")

        # Print the results for verification
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)


2026-03-28 01:36:30,348 - INFO - Starting MLflow run...
2026-03-28 01:36:31,499 - INFO - Logging preprocessing parameters...
2026-03-28 01:36:32,779 - INFO - Initializing Logistic Regression model...
2026-03-28 01:36:32,780 - INFO - Fitting the model...
2026-03-28 01:36:32,800 - INFO - Model training complete.
2026-03-28 01:36:32,800 - INFO - Logging model parameters...
2026-03-28 01:36:33,197 - INFO - Making predictions...
2026-03-28 01:36:33,198 - INFO - Calculating evaluation metrics...
2026-03-28 01:36:33,204 - INFO - Logging evaluation metrics...
2026-03-28 01:36:34,891 - INFO - Saving and logging the model...
2026/03/28 01:36:34 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/28 01:36:39 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The rec

🏃 View run mysterious-colt-612 at: https://dagshub.com/aryan-Patel-web/production-sentiment-analysis-end-to-end-mlops.mlflow/#/experiments/0/runs/ba4a737b0d46469393b33f6b0a145e97
🧪 View experiment at: https://dagshub.com/aryan-Patel-web/production-sentiment-analysis-end-to-end-mlops.mlflow/#/experiments/0
